### Import

In [1]:
import os
import sys
import re
import numpy as np
import pandas as pd
import datetime as dt
from matplotlib import pyplot as plt 

pd.set_option('display.max_columns', 500)

### General parameters

In [2]:
path_csv  = "../Data/csvExtract/"
path_timeseries = "../Data/Output"

### Read cohort icu stays 

In [3]:
cohort_icustay_id = []

with open("../Data/Cohort/cohort1_stay_id.txt", "r") as f:
    for stay_id in f:
        cohort_icustay_id.append(int(stay_id.strip()))
        
len(cohort_icustay_id)

16149

### Demographics

In [4]:
cohort_df = pd.read_csv(path_csv + "patients.csv")
cohort_df[['patientunitstayid']] = cohort_df[['patientunitstayid']].astype(int)

In [5]:
cohort_df['admissionheight'], cohort_df['admissionweight'] = np.where((cohort_df['admissionweight']>=100) & (cohort_df['admissionheight']>25) & (cohort_df['admissionheight']<=100) & (np.abs(cohort_df['admissionheight']-cohort_df['admissionweight']) >=20),(cohort_df['admissionweight'],cohort_df['admissionheight']),(cohort_df['admissionheight'],cohort_df['admissionweight']))

cohort_df.loc[cohort_df['admissionheight'] <= 0.30 , 'admissionheight'] = np.nan
cohort_df.loc[cohort_df['admissionheight'] <= 2.5  , 'admissionheight'] = cohort_df['admissionheight'] * 100
cohort_df.loc[cohort_df['admissionheight'] <= 10   , 'admissionheight'] = np.nan
cohort_df.loc[cohort_df['admissionheight'] <= 25   , 'admissionheight'] = cohort_df['admissionheight'] * 100
cohort_df.loc[cohort_df['admissionweight'] <  20   , 'admissionweight'] = np.nan
cohort_df.loc[cohort_df['admissionweight'] >= 300  , 'admissionweight'] = np.nan

In [6]:
cohort_df.head(3)

### Apache

In [7]:
apache_df = pd.read_csv(path_csv + "apache.csv")
apache_df[['patientunitstayid']] = apache_df[['patientunitstayid']].astype(int)
apache_df.head(3)

### Comorbidities

In [8]:
comorbidities = pd.read_csv(path_csv + "comorbidities.csv")
comorbidities[['patientunitstayid']] = comorbidities[['patientunitstayid']].astype(int)
comorbidities = comorbidities.fillna(0)
comorbidities = comorbidities[comorbidities.patientunitstayid.isin(cohort_icustay_id)]
comorbidities = comorbidities.reset_index(drop=True)
comorbidities.head(3)

### ICD-9

In [9]:
icd9 = pd.read_csv(path_csv + "diagnose_icd9.csv")
icd9[['patientunitstayid']] = icd9[['patientunitstayid']].astype(int)
icd9 = icd9[icd9.patientunitstayid.isin(cohort_icustay_id)]

In [10]:
icd9['itemname'] = 'ICD-9' 
icd9.rename(index=str, columns={'diagnosisoffset': 'itemoffset', "ICD-9": "itemvalue"}, inplace=True)
icd9 = icd9[['patientunitstayid', 'itemoffset', 'itemname', 'itemvalue']]
icd9 = icd9.reset_index(drop=True)

In [11]:
icd9.head(3)

### ICD-10

In [12]:
icd10 = pd.read_csv(path_csv + "diagnose_icd10.csv")
icd10[['patientunitstayid']] = icd10[['patientunitstayid']].astype(int)
icd10 = icd10[icd10.patientunitstayid.isin(cohort_icustay_id)]

In [13]:
icd10_embedd = icd10[['patientunitstayid', 'diagnosisoffset', 'ICD-10_Embedding']]
icd10 = icd10[['patientunitstayid', 'diagnosisoffset', 'ICD-10']]

In [14]:
icd10['itemname'] = 'ICD-10'
icd10.rename(index=str, columns={'diagnosisoffset': 'itemoffset', "ICD-10": "itemvalue"}, inplace=True)
icd10 = icd10[['patientunitstayid', 'itemoffset', 'itemname', 'itemvalue']]
icd10 = icd10.reset_index(drop=True)

In [15]:
icd10_embedd['itemname'] = 'ICD-10_Embedding'
icd10_embedd.rename(index=str, columns={'diagnosisoffset': 'itemoffset', "ICD-10_Embedding": "itemvalue"}, inplace=True)

In [16]:
icd10_embedd = icd10_embedd[['patientunitstayid', 'itemoffset', 'itemname', 'itemvalue']]
icd10_embedd = icd10_embedd.reset_index(drop=True)

In [17]:
icd10.head(3)

In [18]:
icd10_embedd.head(3)

### Elixhauser

In [19]:
elixhauser = pd.read_csv(path_csv + "elixhauser.csv")
elixhauser[['patientunitstayid']] = elixhauser[['patientunitstayid']].astype(int)
elixhauser = elixhauser[elixhauser.patientunitstayid.isin(cohort_icustay_id)]

In [20]:
elixhauser_comorbidity = elixhauser[['patientunitstayid', 'diagnosisoffset', 'comorbidity_lst']].copy()
elixhauser_readmission = elixhauser[['patientunitstayid', 'diagnosisoffset', 'readmission_scr']].copy()
elixhauser_mortalityrisks = elixhauser[['patientunitstayid', 'diagnosisoffset', 'mortality_risks_scr']].copy()

elixhauser_comorbidity['itemname'] = 'elixhauser_comorbidity'
elixhauser_readmission['itemname'] = 'elixhauser_readmission'
elixhauser_mortalityrisks['itemname'] = 'elixhauser_mortalityrisks'

elixhauser_comorbidity.rename(index=str, columns={'diagnosisoffset': 'itemoffset', 
                                                  "comorbidity_lst": "itemvalue"}, inplace=True)
elixhauser_readmission.rename(index=str, columns={'diagnosisoffset': 'itemoffset', 
                                                  "readmission_scr": "itemvalue"}, inplace=True)
elixhauser_mortalityrisks.rename(index=str, columns={'diagnosisoffset': 'itemoffset', 
                                                     "mortality_risks_scr": "itemvalue"}, inplace=True)

elixhauser_comorbidity = elixhauser_comorbidity[['patientunitstayid', 'itemoffset', 'itemname', 'itemvalue']]
elixhauser_readmission = elixhauser_readmission[['patientunitstayid', 'itemoffset', 'itemname', 'itemvalue']]
elixhauser_mortalityrisks = elixhauser_mortalityrisks[['patientunitstayid', 'itemoffset', 'itemname', 'itemvalue']]

elixhauser_comorbidity = elixhauser_comorbidity.reset_index(drop=True)
elixhauser_readmission = elixhauser_readmission.reset_index(drop=True)
elixhauser_mortalityrisks = elixhauser_mortalityrisks.reset_index(drop=True)

In [21]:
elixhauser_comorbidity.head(2)

In [22]:
elixhauser_readmission.head(2)

In [23]:
elixhauser_mortalityrisks.head(2)

### pastHistory 

In [24]:
pastHistory = pd.read_csv(path_csv + "pastHistory.csv")
pastHistory[['patientunitstayid']] = pastHistory[['patientunitstayid']].astype(int)
pastHistory.head(3)

### vitalAperiodic

In [25]:
vitalAperiodic = pd.read_csv(path_csv + "vitalAperiodic.csv")
vitalAperiodic[['patientunitstayid']] = vitalAperiodic[['patientunitstayid']].astype(int)

In [26]:
vitalAperiodic.loc[vitalAperiodic['itemname'] == 'noninvasivemean',      'itemname'] = 'Non-Invasive BP Mean'
vitalAperiodic.loc[vitalAperiodic['itemname'] == 'noninvasivesystolic',  'itemname'] = 'Non-Invasive BP Systolic'
vitalAperiodic.loc[vitalAperiodic['itemname'] == 'noninvasivediastolic', 'itemname'] = 'Non-Invasive BP Diastolic'

In [27]:
vitalAperiodic.head(3)

### vitalPeriodic

In [28]:
vitalPeriodic = pd.read_csv(path_csv + "vitalPeriodic.csv")
vitalPeriodic[['patientunitstayid']] = vitalPeriodic[['patientunitstayid']].astype(int)

In [29]:
vitalPeriodic.loc[vitalPeriodic['itemname'] == 'temperature',         'itemname'] = 'Temperature (C)'
vitalPeriodic.loc[vitalPeriodic['itemname'] == 'heartrate',           'itemname'] = 'Heart Rate'
vitalPeriodic.loc[vitalPeriodic['itemname'] == 'respiration',         'itemname'] = 'Respiratory Rate'
vitalPeriodic.loc[vitalPeriodic['itemname'] == 'cvp',                 'itemname'] = 'CVP'
vitalPeriodic.loc[vitalPeriodic['itemname'] == 'etco2',               'itemname'] = 'ETCO2'
vitalPeriodic.loc[vitalPeriodic['itemname'] == 'systemicsystolic',    'itemname'] = 'Invasive BP Systolic'   
vitalPeriodic.loc[vitalPeriodic['itemname'] == 'systemicdiastolic',   'itemname'] = 'Invasive BP Diastolic'  
vitalPeriodic.loc[vitalPeriodic['itemname'] == 'systemicmean',        'itemname'] = 'Invasive BP Mean'
vitalPeriodic.loc[vitalPeriodic['itemname'] == 'pasystolic',          'itemname'] = 'PA Systolic'    
vitalPeriodic.loc[vitalPeriodic['itemname'] == 'padiastolic',         'itemname'] = 'PA Diastolic'   
vitalPeriodic.loc[vitalPeriodic['itemname'] == 'pamean',              'itemname'] = 'PA Mean'
vitalPeriodic.loc[vitalPeriodic['itemname'] == 'sao2',                'itemname'] = 'SpO2'
vitalPeriodic.loc[vitalPeriodic['itemname'] == 'st1',                 'itemname'] = 'ST1'
vitalPeriodic.loc[vitalPeriodic['itemname'] == 'st2',                 'itemname'] = 'ST2'
vitalPeriodic.loc[vitalPeriodic['itemname'] == 'st3',                 'itemname'] = 'ST3'

In [30]:
vitalPeriodic.head(3)

### Lab

In [31]:
lab = pd.read_csv(path_csv + "lab.csv")
lab[['patientunitstayid']] = lab[['patientunitstayid']].astype(int)
lab.head(2)

In [32]:
lab_var = [
    
'Temperature',
'Respiratory Rate',
'PT',
'PTT', 
'PT - INR',
'pH',
'lactate',
'LDH', 
'Base Excess',    
'anion gap',  
'bicarbonate', 'HCO3',     
'creatinine',     
'Hct', 
'Hgb',    
'total bilirubin', 
'direct bilirubin',
'MPV',
'MCV',
'MCH',
'MCHC',   
'RDW', 
'RBC',     
'WBC x 1000',    
'platelets x 1000',   
'glucose', 'bedside glucose',    
'ammonia',
'magnesium',    
'phosphate',    
'alkaline phos.',    
'potassium',    
'sodium',    
'chloride',
'calcium',  
'ionized calcium',
'total cholesterol',    
'CRP', 'CRP-hs', 
'paO2',
'paCO2',   
'ALT (SGPT)', 
'AST (SGOT)',
'-bands',
'-polys',
'amylase',
'lipase',
'-lymphs',     
'-monos',    
'-eos',
'-basos',    
'LPM O2', 
'O2 Content',    
'O2 Sat (%)',    
'Total CO2',
'albumin',    
'troponin - T', 
'troponin - I',    
'Vancomycin - peak',
'Vancomycin - trough',
'Vancomycin - random',    
'triglycerides',    
'fibrinogen', 
'transferrin',
'Ferritin',    
'cortisol',
'total protein',
'PEEP',
'TV',    
'Vent Rate',
'FiO2',
'BUN',
'TSH',
'Pressure Support',
'Pressure Control', 
'Peak Airway/Pressure',
]

In [33]:
lab = lab[lab.itemname.isin(lab_var)]

In [34]:
lab.loc[lab['itemname'] == 'Temperature', 'itemname'] = 'Temperature (C)'

lab.loc[lab['itemname'] == 'bicarbonate', 'itemname'] = 'Bicarbonate'
lab.loc[lab['itemname'] == 'HCO3', 'itemname'] = 'Bicarbonate'

lab.loc[lab['itemname'] == 'glucose', 'itemname'] = 'Glucose'
lab.loc[lab['itemname'] == 'bedside glucose', 'itemname'] = 'Glucose'

lab.loc[lab['itemname'] == 'CRP', 'itemname'] = 'C-Reactive Protein'
lab.loc[lab['itemname'] == 'CRP-hs', 'itemname'] = 'C-Reactive Protein'

lab.loc[lab['itemname'] == 'O2 Sat (%)', 'itemname'] = 'O2 Saturation'

lab.loc[lab['itemname'] == 'TV', 'itemname'] = 'Tidal Volume'

### Check for Units

In [35]:
lab_unit_df = lab[['itemname', 'labmeasurenamesystem']]
lab_unit_df = lab_unit_df.drop_duplicates()
lab_unit_df = lab_unit_df.groupby(['itemname'])['labmeasurenamesystem'].apply(list).reset_index(name='list_unit')
lab_unit_df['len'] = lab_unit_df.apply(lambda x: len(x['list_unit']), axis=1)
lab_unit_df = lab_unit_df[lab_unit_df.len > 1].reset_index(drop=True)
lab_unit_df.head(3)

,itemname,list_unit,len
0,Total CO2,"[mmol/L, mm/L, MMOL/L]",3
1,anion gap,"[nan, mmol/L, mEq/L, mEQ/L, MMOL/L]",5
2,pH,"[nan, [pH]]",2


In [36]:
lab.drop(columns=['labmeasurenamesystem'], inplace=True)

In [37]:
lab.head(3)

### nurseCharting

In [38]:
nurseCharting = pd.read_csv(path_csv + "nurseCharting.csv")
nurseCharting[['patientunitstayid']] = nurseCharting[['patientunitstayid']].astype(int)

In [39]:
chart_var = [ 
    
'Heart Rate',  
'Temperature (F)', 
'Temperature (C)',
'Respiratory Rate', 
'Non-Invasive BP Mean',
'Non-Invasive BP Systolic',
'Non-Invasive BP Diastolic',    
'Invasive BP Mean',
'Invasive BP Diastolic',
'Invasive BP Systolic',
'PA Mean',
'PA Diastolic', 
'PA Systolic',
'Pain Goal',
'Pain Score',
'Pain Present',  
'GCS Total',  'Score (Glasgow Coma Scale)',
'Motor', 'Motor Response', 'Best Motor Response', 
'Verbal', 'Verbal Response','Best Verbal Response', 
'Eyes', 'Eye Opening', 'Best Eye Response',     
'RASS',
'Fall Risk',  
'Delirium Score',
'Delirium Scale',
'Symptoms of Delirium Present',    
'Sedation Goal',
'Sedation Score', 'SEDATION SCORE',    
'Flow Rate',  
'SpO2', 
'O2 Saturation',
'SVO2',
'Pulse',    
'Bedside Glucose',   
'End Tidal CO2', 
'CVP', 'CVP (mmHg)',
'O2 L/%',
'O2 Admin Device',
'MAP (mmHg)', 'Arterial Line MAP (mmHg)', 
]

In [40]:
# 'CI', 'PVR', 'ICP', 'CPP', 'P.O.', 'CO',  'SVR', 'SV', 'PAOP', 'SVRI', 'IAP', 'PVRI', 'PR', 'QRS', 
# 'QTc', 'QT', 'Pulse Ox  Mode', 'Temperature Location', 'Level of Assistance', 'Electrolyte Replacement',
# 'CV/ PV Assessment', 'Pain Assessment', 'Genitourinary Assessment', 
# 'Mental Status Assessment', 'Patient s Comfort/Function (Pain) GOAL At Rest', 'Respiratory Assessment',
# 'Integumentary Assessment', 'Gastrointestinal Assessment', 'Musculoskeletal Assessment', 
# 'Neurological Assessment', 'Eye, Ear, Nose, Throat Assessment'

In [41]:
nurseCharting = nurseCharting[nurseCharting.itemname.isin(chart_var)]

In [42]:
nurseCharting.loc[nurseCharting['itemname'] == 'Temperature (F)' , 'itemvalue'] = (nurseCharting[nurseCharting['itemname'] == 'Temperature (F)'].itemvalue.astype(float) - 32) * 5 / 9

In [43]:
nurseCharting.loc[nurseCharting['itemname'] == 'Temperature (F)', 'itemname'] = 'Temperature (C)'

nurseCharting.loc[nurseCharting['itemname'] == 'Score (Glasgow Coma Scale)', 'itemname'] = 'GCS Total'
nurseCharting.loc[nurseCharting['itemname'] == 'Motor Response', 'itemname'] = 'Motor'
nurseCharting.loc[nurseCharting['itemname'] == 'Best Motor Response', 'itemname'] = 'Motor'
nurseCharting.loc[nurseCharting['itemname'] == 'Verbal Response', 'itemname'] = 'Verbal'
nurseCharting.loc[nurseCharting['itemname'] == 'Best Verbal Response', 'itemname'] = 'Verbal'
nurseCharting.loc[nurseCharting['itemname'] == 'Eye Opening', 'itemname'] = 'Eyes'
nurseCharting.loc[nurseCharting['itemname'] == 'Best Eye Response', 'itemname'] = 'Eyes'
nurseCharting.loc[nurseCharting['itemname'] == 'SEDATION SCORE', 'itemname'] = 'Sedation Score'

nurseCharting.loc[nurseCharting['itemname'] == 'Bedside Glucose', 'itemname'] = 'Glucose'
nurseCharting.loc[nurseCharting['itemname'] == 'CVP (mmHg)', 'itemname'] = 'CVP'
nurseCharting.loc[nurseCharting['itemname'] == 'Arterial Line MAP (mmHg)', 'itemname'] = 'MAP (mmHg)'
nurseCharting.loc[nurseCharting['itemname'] == 'End Tidal CO2', 'itemname'] = 'ETCO2'
nurseCharting.loc[nurseCharting['itemname'] == 'O2 L/%', 'itemname'] = 'LPM O2'

In [44]:
nurseCharting.head(3)

### respiratoryData

In [45]:
respiratoryData = pd.read_csv(path_csv + "respiratoryData.csv")
respiratoryData[['patientunitstayid']] = respiratoryData[['patientunitstayid']].astype(int)

In [46]:
respiratoey_var = [
    
'HR', 
'Total RR', 'Resp Rate Total', 
'RR (patient)', 'RR Spont', 'Spontaneous Respiratory Rate', 
'Exhaled MV',
'Exhaled Vt', 
'Exhaled TV (patient)',
'Exhaled TV (machine)', 
'Spont TV', 'Tidal Volume Observed (VT)', 'Tidal Volume, Delivered',
'Peak Pressure',
'Plateau Pressure', 
'Peak Insp. Pressure',
'Mean Airway Pressure', 
'Inspiratory Flow Rate', 'Insp Flow (l/min)',
'FIO2 (%)', 'FiO2', 'Set Fraction of Inspired Oxygen (FIO2)',
'EtCO2', 'ETCO2',
'SaO2',
'O2 Percentage', 
'Oxygen Flow Rate',
'Ventilator Type',
]

In [47]:
respiratoryData = respiratoryData[respiratoryData.itemname.isin(respiratoey_var)]

In [48]:
respiratoryData.loc[respiratoryData['itemname'] == 'HR', 'itemname'] = 'Heart Rate'

respiratoryData.loc[respiratoryData['itemname'] == 'Total RR', 'itemname'] = 'Total Respiratory Rate'
respiratoryData.loc[respiratoryData['itemname'] == 'Resp Rate Total', 'itemname'] = 'Total Respiratory Rate'

respiratoryData.loc[respiratoryData['itemname'] == 'RR Spont', 'itemname'] = 'Respiratory Rate'
respiratoryData.loc[respiratoryData['itemname'] == 'RR (patient)', 'itemname'] = 'Respiratory Rate'
respiratoryData.loc[respiratoryData['itemname'] == 'Spontaneous Respiratory Rate', 'itemname'] = 'Respiratory Rate'

respiratoryData.loc[respiratoryData['itemname'] == 'Spont TV', 'itemname'] = 'Tidal Volume'
respiratoryData.loc[respiratoryData['itemname'] == 'Tidal Volume, Delivered', 'itemname'] = 'Tidal Volume'
respiratoryData.loc[respiratoryData['itemname'] == 'Tidal Volume Observed (VT)', 'itemname'] = 'Tidal Volume'

respiratoryData.loc[respiratoryData['itemname'] == 'FIO2 (%)', 'itemname'] = 'FiO2'
respiratoryData.loc[respiratoryData['itemname'] == 'Set Fraction of Inspired Oxygen (FIO2)', 'itemname'] = 'FiO2'

respiratoryData.loc[respiratoryData['itemname'] == 'EtCO2', 'itemname'] = 'ETCO2'

respiratoryData.loc[respiratoryData['itemname'] == 'SaO2', 'itemname'] = 'O2 Saturation'

respiratoryData.loc[respiratoryData['itemname'] == 'Insp Flow (l/min)', 'itemname'] = 'Inspiratory Flow Rate'

In [49]:
respiratoryData.head(3)

### respiratorySetting

In [50]:
respiratorySetting = pd.read_csv(path_csv + "respiratorySetting.csv")
respiratorySetting[['patientunitstayid']] = respiratorySetting[['patientunitstayid']].astype(int)

In [51]:
respiratorySetting.loc[respiratorySetting['itemname'] == 'FiO2', 'itemname'] = 'FiO2 (Set)'
respiratorySetting.loc[respiratorySetting['itemname'] == 'Pressure Support', 'itemname'] = 'Pressure Support (Set)'
respiratorySetting.loc[respiratorySetting['itemname'] == 'PEEP', 'itemname'] = 'PEEP (Set)'
respiratorySetting.loc[respiratorySetting['itemname'] == 'LPM O2', 'itemname'] = 'LPM O2 (Set)'
respiratorySetting.loc[respiratorySetting['itemname'] == 'Vent Rate', 'itemname'] = 'Vent Rate (Set)'
respiratorySetting.loc[respiratorySetting['itemname'] == 'Tidal Volume (set)', 'itemname'] = 'Tidal Volume (Set)'
respiratorySetting.loc[respiratorySetting['itemname'] == 'TV/kg IBW', 'itemname'] = 'TV/kg IBW (Set)'
respiratorySetting.loc[respiratorySetting['itemname'] == 'PEEP/CPAP', 'itemname'] = 'PEEP/CPAP (Set)'
respiratorySetting.loc[respiratorySetting['itemname'] == 'Flow Sensitivity', 'itemname'] = 'Flow Sensitivity (Set)'
respiratorySetting.loc[respiratorySetting['itemname'] == 'Peak Flow', 'itemname'] = 'Peak Flow (Set)'

In [52]:
respiratorySetting.head(3)

### intakeOutput

In [53]:
intakeOutput = pd.read_csv(path_csv + "intakeOutput.csv")
intakeOutput[['patientunitstayid']] = intakeOutput[['patientunitstayid']].astype(int)
intakeOutput = intakeOutput.drop_duplicates().reset_index(drop=True)

In [54]:
intakeOutput.loc[intakeOutput.itemname == 'Bodyweight (lb)', 'itemvalue'] = intakeOutput['itemvalue'] * 0.45

In [55]:
intakeOutput.loc[intakeOutput['itemname'] == 'Bodyweight (lb)', 'itemname'] = 'Bodyweight'
intakeOutput.loc[intakeOutput['itemname'] == 'Bodyweight (kg)', 'itemname'] = 'Bodyweight'

In [56]:
intakeOutput.head(3)

### infusionDrug

In [57]:
infusionDrug = pd.read_csv(path_csv + "infusionDrug.csv")
infusionDrug[['patientunitstayid']] = infusionDrug[['patientunitstayid']].astype(int)
infusionDrug = infusionDrug.drop_duplicates().reset_index(drop=True)

In [58]:
infusionDrug.head(3)

### medication

In [59]:
medication = pd.read_csv(path_csv + "medication.csv")
medication[['patientunitstayid']] = medication[['patientunitstayid']].astype(int)
medication = medication.drop_duplicates().reset_index(drop=True)

In [60]:
medication.head(3)

### treatment

In [61]:
treatment = pd.read_csv(path_csv + "treatment.csv")
treatment[['patientunitstayid']] = treatment[['patientunitstayid']].astype(int)
treatment = treatment.drop_duplicates().reset_index(drop=True)

In [62]:
treatment.head(3)

### All Tables

In [63]:
tables = [icd9, icd10, icd10_embedd, elixhauser_comorbidity, elixhauser_readmission, elixhauser_mortalityrisks, 
          pastHistory, vitalPeriodic, vitalAperiodic, lab, nurseCharting, respiratoryData, respiratorySetting, 
          intakeOutput, infusionDrug, medication, treatment]

In [64]:
all_tables = pd.concat(tables)
all_tables[['patientunitstayid']] = all_tables[['patientunitstayid']].astype(int)
all_tables = all_tables.sort_values(by=['patientunitstayid', 'itemoffset'], axis=0)
all_tables.reset_index(inplace=True, drop=True)

In [65]:
all_tables.head(2)

### Save Data

In [66]:
def cohort_stay_id(frame):
    cohort = frame.patientunitstayid.unique()
    return cohort

### ICU Admission Information

In [67]:
def break_up_admission_by_unit_stay(admission, output_path, stayid, verbose=1):
    
    unit_stays = stayid
    nb_unit_stays = unit_stays.shape[0]
    
    for i, icu_stay_id in enumerate(unit_stays):
        
        if verbose:
            sys.stdout.write('\rStayID {0} of {1}...'.format(i+1, nb_unit_stays))
            
        dn = os.path.join(output_path, str(icu_stay_id))
        
        try:
            os.makedirs(dn)  
        except:
            pass

        admission.loc[admission.patientunitstayid == icu_stay_id].to_csv(os.path.join(dn, 'admission.csv'), index=False)
    
    if verbose:
        sys.stdout.write('DONE!\n')

In [ ]:
icu_stayid_adm  = cohort_stay_id(cohort_df)
break_up_admission_by_unit_stay(cohort_df, path_timeseries, icu_stayid_adm, verbose=1)

### Apache 

In [69]:
def break_up_apache_by_unit_stay(apache, output_path, stayid, verbose=1):
    
    unit_stays = stayid
    nb_unit_stays = unit_stays.shape[0]
    
    for i, icu_stay_id in enumerate(unit_stays):
        
        if verbose:
            sys.stdout.write('\rStayID {0} of {1}...'.format(i+1, nb_unit_stays))
            
        dn = os.path.join(output_path, str(icu_stay_id))
        
        try:
            os.makedirs(dn)  
        except:
            pass

        apache.loc[apache.patientunitstayid == icu_stay_id].to_csv(os.path.join(dn, 'apache.csv'), index=False)
    
    if verbose:
        sys.stdout.write('DONE!\n')

In [70]:
icu_stayid_apache  = cohort_stay_id(apache_df)
break_up_apache_by_unit_stay(apache_df, path_timeseries, icu_stayid_apache, verbose=1)

StayID 14405 of 14405...DONE!


### Comorbidities

In [71]:
def break_up_comorbidities_by_unit_stay(comorbidity, output_path, stayid, verbose=1):
    
    unit_stays = stayid
    nb_unit_stays = unit_stays.shape[0]
    
    for i, icu_stay_id in enumerate(unit_stays):
        
        if verbose:
            sys.stdout.write('\rStayID {0} of {1}...'.format(i+1, nb_unit_stays))
            
        dn = os.path.join(output_path, str(icu_stay_id))
        
        try:
            os.makedirs(dn)  
        except:
            pass

        comorbidity.loc[comorbidity.patientunitstayid == icu_stay_id].to_csv(os.path.join(dn, 'comorbidity.csv'), index=False)
    
    if verbose:
        sys.stdout.write('DONE!\n')

In [72]:
icu_stayid_comorbidity  = cohort_stay_id(comorbidities)
break_up_comorbidities_by_unit_stay(comorbidities, path_timeseries, icu_stayid_comorbidity, verbose=1)

StayID 13541 of 13541...DONE!


### All Tables

In [73]:
def break_up_all_tables_by_unit_stay(all_tables, output_path, stayid, verbose=1):
    
    unit_stays = stayid
    nb_unit_stays = unit_stays.shape[0]
    
    for i, icu_stay_id in enumerate(unit_stays):
        
        if verbose:
            sys.stdout.write('\rStayID {0} of {1}...'.format(i+1, nb_unit_stays))
            
        dn = os.path.join(output_path, str(icu_stay_id))
        
        try:
            os.makedirs(dn)  
        except:
            pass

        all_tables.loc[all_tables.patientunitstayid == icu_stay_id].to_csv(os.path.join(dn, 'all_tables.csv'), index=False)
    
    if verbose:
        sys.stdout.write('DONE!\n')

In [74]:
stay_id_all_tables  = cohort_stay_id(all_tables)
break_up_all_tables_by_unit_stay(all_tables, path_timeseries, stay_id_all_tables, verbose=1)

StayID 15990 of 15990...DONE!


### Save list of unique variables

In [75]:
all_variables = [
    
'pastHistory', 'ICD-9', 'ICD-10', 'ICD-10_Embedding', 
'elixhauser_comorbidity', 'elixhauser_readmission', 'elixhauser_mortalityrisks',
 
'Fentanyl_PRC', 'Propofol_PRC', 'Norepinephrine_PRC', 'Insulin_PRC', 'Midazolam_PRC', 'Heparin_PRC', 
'Dexmedetomidine_PRC', 'Amiodarone_PRC', 'Vasopressin_PRC', 'Phenylephrine_PRC', 'Dopamine_PRC', 'Nicardipine_PRC',
'Milrinone_PRC', 'Pantoprazole_PRC', 'Diltiazem_PRC', 'Dobutamine_PRC', 'Nitroglycerin_PRC', 'Epinephrine_PRC',
'Antibiotic_PRC', 'Warfarin_PRC', 'Vasopressors',
    
'Urine_IO', 'Propofol_IO', 'Fentanyl_IO', 'Insulin_IO', 'Heparin_IO', 'Midazolam_IO', 'Dexmedetomidine_IO', 
'Vassopressin_IO', 'Albumin_IO', 'Ceftriaxone_IO', 'Cefazolin_IO', 'Cefepime_IO', 'Ceftazidime_IO', 
'Vancomycin_IO', 'Clindamycin_IO', 'Metronidazole_IO', 'Meropenem_IO', 'Acyclovir_IO', 'Azithromycin_IO', 
'Levofloxacin_IO', 'Micafungin_IO', 'Fluconazole_IO', 'Thiamine_IO', 'Dobutamine_IO', 'Milrinone_IO', 
'Fluids_IO', 'OralIntake_IO', 'P.O._IO', 'SodiumChloride_IO', 'IVPB_IO', 'Stool_IO', 'Crystalloids_IO',
'NSIVF_IO', 'Norepinephrine_IO', 'Amiodarone_IO', 'Phenylephrine_IO', 'Epinephrine_IO', 'Nicardipine_IO',
'Pantoprazole_IO', 'Diltiazem_IO', 'Nitroglycerin_IO',
 
'Non-Invasive BP Mean', 'Non-Invasive BP Systolic', 'Non-Invasive BP Diastolic', 
'Invasive BP Systolic', 'Invasive BP Diastolic', 'Invasive BP Mean', 'PA Systolic', 'PA Diastolic', 'PA Mean', 
'Temperature (C)', 'Heart Rate', 'Respiratory Rate', 'CVP', 'ETCO2', 'ST1', 'ST2', 'ST3',
 
'PT', 'PTT', 'PT - INR', 'pH', 'lactate', 'LDH',  'Base Excess', 'anion gap', 'Bicarbonate', 'creatinine',     
'Hct', 'Hgb', 'total bilirubin', 'direct bilirubin', 'MPV', 'MCV', 'MCH', 'MCHC',   
'RDW', 'RBC', 'WBC x 1000', 'platelets x 1000', 'Glucose', 'ammonia', 'magnesium', 'phosphate', 'alkaline phos.',    
'potassium',  'sodium', 'chloride', 'calcium', 'ionized calcium', 'total cholesterol', 'C-Reactive Protein',
'paO2', 'paCO2', 'ALT (SGPT)', 'AST (SGOT)', '-bands', '-polys', 'amylase', 'lipase', '-lymphs', '-monos',    
'-eos', '-basos', 'LPM O2', 'O2 Content', 'O2 Saturation', 'Total CO2', 'albumin', 'troponin - T', 'troponin - I',    
'Vancomycin - peak', 'Vancomycin - trough', 'Vancomycin - random', 'triglycerides', 'fibrinogen', 'transferrin',
'Ferritin', 'cortisol', 'total protein', 'PEEP', 'Tidal Volume', 'Vent Rate', 'FiO2', 'BUN', 'TSH',
'Pressure Support', 'Pressure Control', 'Peak Airway/Pressure', 'Pain Goal', 'Pain Score', 'Pain Present',  
'GCS Total', 'Motor', 'Verbal', 'Eyes', 'RASS', 'Fall Risk',  'Delirium Score', 'Delirium Scale',
'Symptoms of Delirium Present', 'Sedation Goal', 'Sedation Score', 'Flow Rate', 'SpO2', 'SVO2',
'Pulse', 'O2 Admin Device', 'MAP (mmHg)', 'Total Respiratory Rate', 'Exhaled MV', 'Exhaled Vt', 
'Exhaled TV (patient)', 'Exhaled TV (machine)', 'Peak Pressure', 'Plateau Pressure', 'Peak Insp. Pressure', 
'Mean Airway Pressure', 'Inspiratory Flow Rate', 'O2 Percentage', 'Oxygen Flow Rate',
'Ventilator Type', 
    
'FiO2 (Set)', 'Pressure Support (Set)', 'PEEP (Set)', 'LPM O2 (Set)', 'Vent Rate (Set)', 'Tidal Volume (Set)',
'TV/kg IBW (Set)', 'PEEP/CPAP (Set)', 'Flow Sensitivity (Set)', 'Peak Flow (Set)', 'Bodyweight'

]

cat_int_value = ['Pain Goal', 'Pain Score', 'GCS Total', 'Motor', 'Verbal', 'Eyes', 'RASS', 'Fall Risk',
                 'Delirium Score', 'Symptoms of Delirium Present', 'Sedation Goal', 'Sedation Score', 
                 'Ventilator Type', 'Pain Present']

cat_text_value = ['Delirium Scale', 'O2 Admin Device']

In [76]:
with open("../Data/csvExtract/variables.txt", "w") as f:
    for variable in all_variables:
        f.write(variable +"\n")